Setup & Imports

In [ ]:
import sys
from pathlib import Path

sys.path.append("..")

from src.preprocessing.preprocessor import TextPreprocessor
from src.extraction.extractor import extract_text
from src.utils.config_loader import load_config

config = load_config()
preprocessor = TextPreprocessor()

print("=== PREPROCESSOR INITIALIZED ===")
print(f"Active NLP Engine Mode : {preprocessor.method.upper()}")
print(f"spaCy Model            : {config['preprocessing']['spacy_model']}")
print(f"Total Stopwords Loaded : {len(preprocessor.stopwords)}")
print(f"Protected Terms Count  : {len(preprocessor.protected_terms)}")

Unit Testing Core Preprocessing Functions

In [ ]:
sample_raw_text = (
    "The cell membrane is a thin, flexible barrier protecting the cell. "
    "H2O and CO2 molecules diffuse across it easily."
)

print("--- RAW TEXT ---")
print(sample_raw_text)

# Token list output
tokens = preprocessor.preprocess(sample_raw_text)
print("\n--- TOKEN LIST OUTPUT (preprocess) ---")
print(tokens)

# Cleaned string output
cleaned_string = preprocessor.preprocess_to_string(sample_raw_text)
print("\n--- CLEANED STRING OUTPUT (preprocess_to_string) ---")
print(cleaned_string)

# Edge case: Empty / Whitespace strings
print("\n--- EDGE CASE CHECKS ---")
print("Empty string output      :", preprocessor.preprocess(""))
print("Whitespace string output :", preprocessor.preprocess("     \n\t  "))

Testing Protected Technical Terms & N-Gram Generation

In [ ]:
# 1. Verify protected STEM terminology survives filtering
biology_stem_text = "The cellular respiration produces ATP, DNA, RNA, H2O, and CO2 under neutral pH."
stem_tokens = preprocessor.preprocess(biology_stem_text)

print("--- PROTECTED TERMS RETENTION TEST ---")
print(f"Original Text : {biology_stem_text}")
print(f"Tokens Output : {stem_tokens}")

protected_survived = [t for t in preprocessor.protected_terms if t in stem_tokens]
print(f"\nProtected Terms Retained : {protected_survived}")

# 2. Bigrams / Trigrams Generation
bigrams = preprocessor.generate_ngrams(tokens, n=2)
trigrams = preprocessor.generate_ngrams(tokens, n=3)

print("\n--- N-GRAMS GENERATION ---")
print(f"Sample Bigrams  (n=2) : {bigrams[:5]}")
print(f"Sample Trigrams (n=3) : {trigrams[:3]}")

Dual-Engine Verification (TF-IDF Filtering vs. Semantic Bypass)

In [ ]:
# Inspect behavior difference between Semantic (transformer) and TF-IDF (lexical) modes
print(f"Current Configured Mode: {preprocessor.method.upper()}")

test_sentence = "The plasma membrane is NOT permeable to all large molecules."

if preprocessor.method == "semantic":
    print("Semantic Mode active: Stopwords (like 'not', 'is', 'to') are preserved for transformer context.")
else:
    print("TF-IDF Mode active: Stopwords stripped and lemmas extracted for exact lexical overlap.")

processed_demo = preprocessor.preprocess_to_string(test_sentence)
print(f"\nOriginal  : {test_sentence}")
print(f"Processed : {processed_demo}")

Real Extracted Data to Disk Saving Pipeline

In [ ]:
import os

samples_dir = Path("../data/samples")
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

print("=== BATCH PREPROCESSING & SAVING TO data/processed/ ===")

for fname in sorted(os.listdir(samples_dir)):
    if fname.startswith("."):
        continue
        
    fpath = samples_dir / fname
    extraction = extract_text(str(fpath))
    
    if "error" in extraction["metadata"]:
        print(f"Skipping {fname} due to extraction error.")
        continue
        
    raw_text = extraction["text"]
    tokens = preprocessor.preprocess(raw_text)
    
    # Save tokens to data/processed/<stem>_preprocessed.txt
    preprocessor.save_preprocessed(tokens, original_filename=str(fpath), output_dir=str(processed_dir))
    
    print(f"{fname:<20} | Raw Chars: {len(raw_text):<5} | Clean Tokens: {len(tokens):<4} -> Saved ✅")